# Featuresmith Tutorial: 06 — End-to-End ML Dataset Validation Workflow

Build a production-ready ML dataset validation pipeline connecting dataset loading, automated review, readiness gating, error handling, and JSON export.

---


## 1. Building a Production Pre-Training Gate
This notebook demonstrates how to build an end-to-end dataset validation pipeline in Python that can gate model training jobs automatically.

### Step 1: Complete Pipeline Function

In [1]:
import os

import featuresmith as fs
from featuresmith.core.exceptions import ConnectorError


def validate_and_gate_dataset(
    file_path: str, target_col: str, min_score: float = 80.0
) -> bool:
    print(f"=== Validating Dataset: {file_path} ===")
    try:
        dataset = fs.load(file_path)
    except ConnectorError as err:
        print(f"❌ GATE FAILED: Connector Error loading {file_path}: {err}")
        return False

    review_res = fs.review(dataset, target_column=target_col)
    scorecard = fs.score(review_res)

    overall_score = scorecard.overall if scorecard else 0.0
    print(f"ML Readiness Score: {overall_score:.1f} / 100 (Threshold: {min_score:.1f})")

    all_findings = [f for s in review_res.sections for f in s.findings]
    critical_findings = [
        f
        for f in all_findings
        if f.severity == "critical"
        or (hasattr(f.severity, "value") and f.severity.value == "critical")
    ]

    if critical_findings:
        print(f"❌ GATE FAILED: {len(critical_findings)} critical finding(s) detected!")
        for f in critical_findings:
            print(f"   - [{f.rule_id}] {f.title}")
        return False

    if overall_score < min_score:
        print(
            f"❌ GATE FAILED: Readiness score {overall_score:.1f} is below minimum threshold {min_score:.1f}."
        )
        return False

    print("✅ GATE PASSED: Dataset is clean and ready for model training.")
    return True


titanic_path = os.path.join("..", "data", "processed", "titanic.csv")
passed = validate_and_gate_dataset(titanic_path, target_col="survived", min_score=80.0)
print(f"Pipeline Gate Result: {passed}")

=== Validating Dataset: ..\data\processed\titanic.csv ===
ML Readiness Score: 86.9 / 100 (Threshold: 80.0)
❌ GATE FAILED: 1 critical finding(s) detected!
   - [quality.missing_value_threshold] High missing values in column 'cabin'
Pipeline Gate Result: False


### Key Takeaways & Connection to Next Tutorial
- Featuresmith enables programmatic dataset quality gating in production pipelines.
- Zero external network dependencies ensure data remains 100% private and secure.

**Next Tutorial**: In `07_custom_rules_and_extensions.ipynb`, we learn how to extend `BaseRule` to write custom domain-specific validation rules.